# 7.8 Business Applications of Clustering

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 7: Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-07-clustering-techniques/07.08_business-applications-of-clustering.ipynb)

Companion code for Section 7.8 of the book. Run the cells in order from top to bottom.

**Example 7.8.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings; warnings.filterwarnings('ignore')

# ── Simulate e-commerce customer data ────────────────────────────
np.random.seed(42)
n = 2000

# Create realistic customer segments
segments = {
    'Champions':        {'n':300,  'recency':(5,30),   'frequency':(15,30), 'monetary':(500,2000)},
    'Loyal':            {'n':400,  'recency':(10,60),  'frequency':(8,20),  'monetary':(200,800)},
    'At-Risk':          {'n':500,  'recency':(60,120), 'frequency':(5,15),  'monetary':(150,500)},
    'Lost Customers':   {'n':400,  'recency':(120,365),'frequency':(1,5),   'monetary':(50,200)},
    'New Customers':    {'n':400,  'recency':(1,30),   'frequency':(1,3),   'monetary':(50,300)},
}

rows = []
true_labels = []
for seg_name, seg in segments.items():
    n_seg = seg['n']
    r = np.random.uniform(*seg['recency'],    n_seg)
    f = np.random.uniform(*seg['frequency'],  n_seg)
    m = np.random.uniform(*seg['monetary'],   n_seg)
    age   = np.random.randint(18, 70, n_seg)
    clv   = m * f * np.random.uniform(0.8, 1.2, n_seg)  # Customer Lifetime Value proxy
    for i in range(n_seg):
        rows.append({'recency_days': r[i], 'frequency': f[i],
                     'monetary_avg': m[i], 'age': age[i], 'clv_score': clv[i]})
        true_labels.append(seg_name)

df = pd.DataFrame(rows)
true_labels = np.array(true_labels)

print('Customer Dataset Overview:')
print(df.describe().round(2))

# ── RFM Feature Engineering ───────────────────────────────────────
# Recency: lower is better → invert
df['recency_score']   = 1 / (df['recency_days'] + 1)
df['frequency_score'] = np.log1p(df['frequency'])
df['monetary_score']  = np.log1p(df['monetary_avg'])
df['clv_score']       = np.log1p(df['clv_score'])

features = ['recency_score','frequency_score','monetary_score','age','clv_score']
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Find optimal k ────────────────────────────────────────────────
k_scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)
    k_scores[k] = {'inertia': km.inertia_,
                   'silhouette': silhouette_score(X_scaled, labels)}

best_k = max(k_scores, key=lambda k: k_scores[k]['silhouette'])
print(f'\nOptimal k = {best_k} (silhouette = {k_scores[best_k]["silhouette"]:.4f})')

# ── Final segmentation ────────────────────────────────────────────
km_final = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42)
df['segment'] = km_final.fit_predict(X_scaled)

# Interpret and name segments by their characteristics
seg_profiles = df.groupby('segment').agg(
    n_customers = ('recency_days','count'),
    mean_recency = ('recency_days','mean'),
    mean_frequency = ('frequency','mean'),
    mean_spend = ('monetary_avg','mean'),
    mean_clv = ('clv_score','mean')
).round(2)

# Name segments by value
seg_profiles['total_value'] = seg_profiles['mean_frequency'] * seg_profiles['mean_spend']
seg_profiles = seg_profiles.sort_values('total_value', ascending=False)
seg_names = ['Champions','Loyal Customers','At-Risk Customers','Lost Customers','New Customers'][:best_k]
seg_profiles['segment_name'] = seg_names

print('\n=== Customer Segment Profiles ===')
print(seg_profiles.to_string())

# ── Business Recommendations ──────────────────────────────────────
recommendations = {
    'Champions':        'Reward with loyalty programs; early access to new products; referral program',
    'Loyal Customers':  'Upsell premium products; ask for reviews; maintain engagement',
    'At-Risk Customers':'Win-back campaign: personalised discount; ask what changed',
    'Lost Customers':   'Reactivation email with significant offer or product updates',
    'New Customers':    'Onboarding campaign; showcase popular products; collect preferences',
}
print('\n=== Business Action Plan ===')
for seg, rec in list(recommendations.items())[:best_k]:
    n_cust = len(df[df['segment'] == seg_profiles.index[seg_names.index(seg)]])
    print(f'\n{seg} ({n_cust} customers):')
    print(f'  Action: {rec}')

# ── Visualisation ─────────────────────────────────────────────────
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)
df['pc1'] = X_2d[:,0]
df['pc2'] = X_2d[:,1]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Customer Segmentation - E-commerce RFM Analysis', fontsize=14, fontweight='bold')

# Cluster scatter (PCA)
ax = axes[0]
palette_seg = ['#E53935','#1976D2','#43A047','#FF6F00','#9C27B0']
for k in range(best_k):
    mask = df['segment'] == k
    seg_label = seg_names[list(seg_profiles.index).index(k)] if k in seg_profiles.index else f'Seg{k}'
    ax.scatter(df[mask].pc1, df[mask].pc2, s=8, alpha=0.6,
              color=palette_seg[k], label=seg_label)
ax.set_title('Customer Segments (PCA 2D)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Segment size pie chart
ax = axes[1]
sizes = df['segment'].value_counts().sort_index()
labels_pie = [seg_names[list(seg_profiles.index).index(i)] for i in sizes.index]
wedges, texts, autotexts = ax.pie(sizes, labels=labels_pie, autopct='%1.1f%%',
                                    colors=palette_seg[:best_k], startangle=90)
ax.set_title('Segment Distribution')

# Segment profile heatmap
ax = axes[2]
profile_data = df.groupby('segment')[['recency_score','frequency_score',
                                       'monetary_score','clv_score']].mean()
profile_norm = (profile_data - profile_data.min()) / (profile_data.max() - profile_data.min())
sns.heatmap(profile_norm, annot=True, fmt='.2f', cmap='RdYlGn',
           ax=ax, linewidths=0.5, cbar_kws={'shrink':0.8})
ax.set_title('Segment Feature Profiles\n(Normalised 0-1)')
ax.set_yticklabels([f'Seg {i}' for i in profile_norm.index], rotation=0)

plt.tight_layout()
plt.savefig('customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

**Example 7.8.2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

np.random.seed(42)

# ── Simulate transaction data with fraud ──────────────────────────
n_normal = 1000
n_fraud  = 30

# Normal transactions: clustered around typical spending patterns
X_normal = np.vstack([
    np.random.multivariate_normal([10, 50],  [[4,2],[2,25]], n_normal//3),
    np.random.multivariate_normal([30, 200], [[16,8],[8,100]], n_normal//3),
    np.random.multivariate_normal([5,  20],  [[1,0.5],[0.5,4]], n_normal//3),
])

# Fraudulent transactions: unusual patterns
X_fraud = np.array([
    *np.random.uniform([0,500], [2,2000], (n_fraud//3, 2)),   # Very high amounts
    *np.random.uniform([100,0], [500,5],  (n_fraud//3, 2)),   # Very frequent small
    *np.random.uniform([-5,-10],[0,0],    (n_fraud//3, 2)),   # Negative/invalid
])

X_all = np.vstack([X_normal, X_fraud])
y_all = np.array([0]*n_normal + [1]*n_fraud)  # 0=normal, 1=fraud
X_sc  = StandardScaler().fit_transform(X_all)

print('=== Anomaly Detection Comparison ===')
print(f'Normal: {n_normal}, Fraud: {n_fraud}')

# Method 1: K-Means distance-based
km = KMeans(n_clusters=3, n_init=10, random_state=42)
km.fit(X_sc)
# Anomaly score = distance to nearest centroid
distances = np.min(np.array([
    np.linalg.norm(X_sc - c, axis=1) for c in km.cluster_centers_
]), axis=0)
threshold_km = np.percentile(distances, 97)   # Top 3% as anomalies
preds_km = (distances > threshold_km).astype(int)

# Method 2: DBSCAN noise points
db = DBSCAN(eps=0.4, min_samples=8)
db_labels = db.fit_predict(X_sc)
preds_db = (db_labels == -1).astype(int)   # Noise = anomaly

# Method 3: Isolation Forest
iso = IsolationForest(contamination=0.03, random_state=42)
preds_iso = (iso.fit_predict(X_sc) == -1).astype(int)

# Evaluation
from sklearn.metrics import precision_score, recall_score, f1_score
print(f'\n{"Method":<22} {"Precision":>11} {"Recall":>10} {"F1":>8} {"Flagged":>8}')
print('='*60)
for name, preds in [('K-Means Distance', preds_km),
                    ('DBSCAN Noise',     preds_db),
                    ('Isolation Forest', preds_iso)]:
    p = precision_score(y_all, preds, zero_division=0)
    r = recall_score(y_all, preds, zero_division=0)
    f = f1_score(y_all, preds, zero_division=0)
    flagged = preds.sum()
    print(f'{name:<22} {p:>11.4f} {r:>10.4f} {f:>8.4f} {flagged:>8}')

# ── Visualise ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Anomaly Detection - K-Means vs DBSCAN vs Isolation Forest',
            fontsize=13, fontweight='bold')

for ax, (name, preds) in zip(axes, [
    ('K-Means Distance', preds_km),
    ('DBSCAN Noise',     preds_db),
    ('Isolation Forest', preds_iso)
]):
    # Normal points
    norm_mask  = preds == 0
    anom_mask  = preds == 1
    true_fraud = y_all == 1

    ax.scatter(X_sc[norm_mask,0], X_sc[norm_mask,1], s=10, alpha=0.5,
              color='#1976D2', label='Normal')
    ax.scatter(X_sc[anom_mask,0], X_sc[anom_mask,1], s=80, alpha=0.9,
              color='#E53935', marker='x', lw=2, label=f'Flagged ({anom_mask.sum()})')

    # True fraud markers
    ax.scatter(X_sc[true_fraud,0], X_sc[true_fraud,1], s=100,
              color='orange', marker='D', alpha=0.8, label=f'True Fraud ({n_fraud})')

    p = precision_score(y_all, preds, zero_division=0)
    r = recall_score(y_all, preds, zero_division=0)
    ax.set_title(f'{name}\nP={p:.2f}  R={r:.2f}')
    ax.set_xlabel('Feature 1 (Amount)'); ax.set_ylabel('Feature 2 (Frequency)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

**Example 7.8.3**

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

# Load 20 Newsgroups - 4 topic categories
categories = ['sci.med','sci.space','comp.graphics','rec.sport.hockey']
newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                 remove=('headers','footers','quotes'),
                                 random_state=42)
X_text = newsgroups.data
y_true = newsgroups.target
category_names = newsgroups.target_names

print(f'Documents: {len(X_text)}')
print(f'Categories: {category_names}')
print(f'Distribution: {dict(zip(*np.unique(y_true, return_counts=True)))}')

# ── TF-IDF Vectorisation ──────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1,2),
    min_df=3,
    sublinear_tf=True   # Apply log(1+tf) - reduces impact of frequent terms
)
X_tfidf = tfidf.fit_transform(X_text)  # Sparse matrix
print(f'\nTF-IDF matrix shape: {X_tfidf.shape}  (sparse: {1-X_tfidf.nnz/np.prod(X_tfidf.shape):.1%} zeros)')

# ── Dimensionality reduction with LSA (SVD) ────────────────────────
# K-Means struggles in very high dimensions - reduce first
svd = TruncatedSVD(n_components=100, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)
print(f'LSA shape: {X_lsa.shape}  (variance explained: {svd.explained_variance_ratio_.sum():.1%})')

# ── K-Means on text ───────────────────────────────────────────────
km_text = MiniBatchKMeans(n_clusters=4, n_init=10, random_state=42)
cluster_labels = km_text.fit_predict(X_lsa)

ari = adjusted_rand_score(y_true, cluster_labels)
nmi = normalized_mutual_info_score(y_true, cluster_labels)
print(f'\nText Clustering Results (k=4):')
print(f'  ARI: {ari:.4f}')
print(f'  NMI: {nmi:.4f}')

# ── Top terms per cluster ─────────────────────────────────────────
# Transform original TF-IDF centroids back to vocabulary space
centroids = svd.inverse_transform(km_text.cluster_centers_)
top_terms_per_cluster = {}

print('\nTop 10 terms per cluster (document topics):')
feature_names = np.array(tfidf.get_feature_names_out())
for k in range(4):
    top_idx = centroids[k].argsort()[::-1][:10]
    top_terms = feature_names[top_idx]
    top_terms_per_cluster[k] = top_terms
    # Which true category does this cluster mostly contain?
    dominant_cat = category_names[np.bincount(y_true[cluster_labels==k]).argmax()]
    print(f'\n  Cluster {k} (most like: {dominant_cat}):')
    print(f'    {" | ".join(top_terms)}')

# ── Cluster-Category matrix ───────────────────────────────────────
print('\nCluster-Category Assignment Matrix:')
import pandas as pd
cross_tab = pd.crosstab(
    pd.Series(cluster_labels, name='Cluster'),
    pd.Series([category_names[y] for y in y_true], name='True Category')
)
print(cross_tab)